# Assignment 6 — Myanmar ↔ Hiero NMT with Marian

- Class: AIE-F (Batch-2)
- Submission: AungCN

Marian runs in Docker (CPU). The pipeline is in `nmt.py`. This notebook runs it and shows the results.

## 1. Setup

In [1]:
MARIAN = "docker run --rm --platform linux/amd64 --entrypoint bash assignment6-marian:cpu -c"
!{MARIAN} "marian --version"

v1.12.43 c9f287d 2025-06-23 19:47:45 +0000


## 2. Data

In [2]:
!wc -l myhi_data/*.my myhi_data/*.hi

     500 myhi_data/test.my
    8500 myhi_data/train.my
    1000 myhi_data/valid.my
     500 myhi_data/test.hi
    8500 myhi_data/train.hi
    1000 myhi_data/valid.hi
   20000 total


In [3]:
!paste myhi_data/train.my myhi_data/train.hi | head -5

မင်းက အေးဆေးတယ်	𓂨𓀀𓎡𓋴𓂝 𓏮𓀀𓐍𓂋
ရိုးသားမှုရှိတယ်	𓏏𓂝𓋴𓂝 𓈙𓆑𓎛𓏭𓂝𓋴𓂝
ကျွန်မ ရိုးသားနိုင်လား။	𓏭𓄿𓎛 𓆑 𓇋𓂝 𓊪𓐍𓎛𓂝𓈙𓈎
ဘယ်လို သက်သာမှုလဲ။	𓏏𓊪𓄿𓈎 𓄿 𓋴𓂝𓂋𓆑𓂝𓅱
ကျွန်မသွားခွင့်ပြုပါ။	𓄿𓂋𓂋𓐍𓏏 𓉔𓂝 𓈎𓐍 𓃀𓐍
paste: stdout: Broken pipe


## 3. Preprocessing

- Myanmar is split into syllables.
- Hiero is kept in two forms: word (`.hi`) and glyph (`.hic`). `▁` marks a word boundary.
- Vocab is built from the training set only.

In [4]:
!python3 nmt.py prep

train  8500 pairs
valid  1000 pairs
test    500 pairs


vocab.my.yml 1093


vocab.hi.yml 2860


vocab.hic.yml 39


In [5]:
!head -3 data/train.my data/train.hi data/train.hic

==> data/train.my <==
မင်း က အေး ဆေး တယ်
ရိုး သား မှု ရှိ တယ်
ကျွန် မ ရိုး သား နိုင် လား ။

==> data/train.hi <==
𓂨𓀀𓎡𓋴𓂝 𓏮𓀀𓐍𓂋
𓏏𓂝𓋴𓂝 𓈙𓆑𓎛𓏭𓂝𓋴𓂝
𓏭𓄿𓎛 𓆑 𓇋𓂝 𓊪𓐍𓎛𓂝𓈙𓈎

==> data/train.hic <==
𓂨 𓀀 𓎡 𓋴 𓂝 ▁ 𓏮 𓀀 𓐍 𓂋
𓏏 𓂝 𓋴 𓂝 ▁ 𓈙 𓆑 𓎛 𓏭 𓂝 𓋴 𓂝
𓏭 𓄿 𓎛 ▁ 𓆑 ▁ 𓇋 𓂝 ▁ 𓊪 𓐍 𓎛 𓂝 𓈙 𓈎


In [6]:
!head -8 data/vocab.my.yml data/vocab.hi.yml data/vocab.hic.yml

==> data/vocab.my.yml <==
</s>: 0
<unk>: 1
။: 2
တယ်: 3
မ: 4
ပါ: 5
က: 6
ကျွန်: 7

==> data/vocab.hi.yml <==
</s>: 0
<unk>: 1
𓆑: 2
𓆑𓈙: 3
𓄿: 4
𓂨𓀀𓎡: 5
𓆑𓈎: 6
𓆑𓉔: 7

==> data/vocab.hic.yml <==
</s>: 0
<unk>: 1
▁: 2
𓂝: 3
𓆑: 4
𓈎: 5
𓄿: 6
𓈙: 7


In [7]:
from pathlib import Path

hi_train = set(Path("data/train.hi").read_text().split())
hi_test = Path("data/test.hi").read_text().split()
oov = sum(w not in hi_train for w in hi_test)
print(f"Hiero test words not in train: {oov}/{len(hi_test)} = {oov / len(hi_test):.1%}")

Hiero test words not in train: 83/1573 = 5.3%


Some Hiero test words are unseen as whole words. Glyph units avoid this, so both forms are tested.

## 4. Models

Two architectures: `s2s` (GRU) and `tf` (Transformer). Settings are in `nmt.py`.

In [8]:
import nmt

for name, opts in nmt.EXPS.items():
    print(name, " ".join(opts), sep="\n  ", end="\n\n")
print("common", " ".join(nmt.COMMON), sep="\n  ")

s2s
  --type s2s --dim-emb 128 --dim-rnn 256 --enc-type bidirectional --enc-cell gru --dec-cell gru --enc-depth 1 --dec-depth 1 --dropout-rnn 0.2 --dropout-src 0.1 --dropout-trg 0.1 --tied-embeddings --layer-normalization --mini-batch 64 --learn-rate 0.001

tf
  --type transformer --dim-emb 256 --transformer-dim-ffn 1024 --enc-depth 2 --dec-depth 2 --transformer-heads 4 --transformer-dropout 0.1 --label-smoothing 0.1 --tied-embeddings --mini-batch 64 --learn-rate 0.0005 --lr-warmup 400 --lr-decay-inv-sqrt 400 --clip-norm 5

common
  --max-length 100 --valid-metrics bleu ce-mean-words perplexity --early-stopping 8 --keep-best --beam-size 4 --normalize 0.6 --valid-mini-batch 32 --quiet-translation --overwrite --seed 1111


## 5. Training

Training takes long on CPU, so it was run from the terminal with `run_all.sh`. The logs are shown here.

In [9]:
!cat run_all.sh

#!/usr/bin/env bash
# Runs the experiments in parallel. Usage: ./run_all.sh base|tune
set -euo pipefail
cd "$(dirname "$0")"

case "${1:-base}" in
  base)
    for dir in my-hi hi-my; do
      for hi in word char; do
        python3 nmt.py train s2s "$dir" --hi "$hi" &
        python3 nmt.py train tf "$dir" --hi "$hi" &
      done
    done
    ;;
  tune)
    for dir in my-hi hi-my; do
      python3 nmt.py train tf-small "$dir" --dim-emb 128 --transformer-dim-ffn 256 --enc-depth 1 --dec-depth 1 --transformer-heads 2 &
      python3 nmt.py train tf-drop "$dir" --transformer-dropout 0.3 &
      python3 nmt.py train tf-deep "$dir" --enc-depth 3 --dec-depth 3 --transformer-dropout 0.3 &
    done
    ;;
esac
wait


In [10]:
import re


def best_valid(model):
    scores = re.findall(r"Ep\. (\d+) : Up\. (\d+) : bleu : ([\d.]+)", Path(f"models/{model}/valid.log").read_text())
    ep, up, bleu = max(scores, key=lambda s: float(s[2]))
    return f"{model:22} best valid BLEU {float(bleu):6.2f} at epoch {ep}, update {up} (last epoch {scores[-1][0]})"


BASE = [f"{m}.{u}.{d}" for d in ("my-hi", "hi-my") for u in ("word", "char") for m in ("s2s", "tf")]
for model in BASE:
    print(best_valid(model))

s2s.word.my-hi         best valid BLEU  35.22 at epoch 22, update 2800 (last epoch 46)
tf.word.my-hi          best valid BLEU  31.96 at epoch 19, update 2400 (last epoch 43)
s2s.char.my-hi         best valid BLEU  60.39 at epoch 46, update 6000 (last epoch 61)
tf.char.my-hi          best valid BLEU  58.20 at epoch 55, update 7200 (last epoch 61)
s2s.word.hi-my         best valid BLEU  49.76 at epoch 55, update 7200 (last epoch 61)
tf.word.hi-my          best valid BLEU  50.58 at epoch 46, update 6000 (last epoch 61)
s2s.char.hi-my         best valid BLEU  45.79 at epoch 61, update 7980 (last epoch 61)
tf.char.hi-my          best valid BLEU  38.61 at epoch 52, update 6800 (last epoch 61)


Valid BLEU for `char` my-hi is on glyphs, so it is not comparable with `word` here. Test BLEU below is always on words.

In [11]:
!tail -6 models/tf.char.my-hi/train.log

[2026-10-10 15:57:47] [valid] Ep. 61 : Up. 7980 : bleu : 57.5332 : stalled 2 times (last best: 58.1989)
[2026-10-10 15:57:50] [valid] Ep. 61 : Up. 7980 : ce-mean-words : 0.795362 : stalled 11 times (last best: 0.78551)
[2026-10-10 15:57:54] [valid] Ep. 61 : Up. 7980 : perplexity : 2.21524 : stalled 11 times (last best: 2.19352)
[2026-10-10 15:57:54] Saving model weights and runtime parameters to models/tf.char.my-hi/model.npz
[2026-10-10 15:57:54] Saving Adam parameters
[2026-10-10 15:57:54] [training] Saving training checkpoint to models/tf.char.my-hi/model.npz and models/tf.char.my-hi/model.npz.optimizer.npz


## 6. Test — Seq2Seq

BLEU is from `multi-bleu.perl`. Hiero is scored on words. Myanmar is scored on syllables.

In [12]:
!rm -f results/bleu.tsv
for d in ("my-hi", "hi-my"):
    for u in ("word", "char"):
        !python3 nmt.py test s2s {d} --hi {u}

s2s.word my-hi: BLEU = 33.78, 59.4/35.8/26.2/25.2 (BP=0.982, ratio=0.982, hyp_len=1545, ref_len=1573)


s2s.char my-hi: BLEU = 27.23, 56.1/32.3/21.2/14.3 (BP=1.000, ratio=1.028, hyp_len=1617, ref_len=1573)


s2s.word hi-my: BLEU = 48.71, 71.6/54.5/41.9/34.7 (BP=0.998, ratio=0.998, hyp_len=3579, ref_len=3585)


s2s.char hi-my: BLEU = 44.75, 69.3/51.8/39.2/32.1 (BP=0.971, ratio=0.971, hyp_len=3481, ref_len=3585)


## 7. Test — Transformer

In [13]:
for d in ("my-hi", "hi-my"):
    for u in ("word", "char"):
        !python3 nmt.py test tf {d} --hi {u}

tf.word my-hi: BLEU = 33.66, 60.2/37.2/28.1/24.2 (BP=0.958, ratio=0.959, hyp_len=1509, ref_len=1573)


tf.char my-hi: BLEU = 26.59, 54.7/30.0/19.5/15.6 (BP=1.000, ratio=1.020, hyp_len=1604, ref_len=1573)


tf.word hi-my: BLEU = 49.65, 73.8/56.9/43.9/36.3 (BP=0.976, ratio=0.977, hyp_len=3501, ref_len=3585)


tf.char hi-my: BLEU = 37.06, 62.4/44.0/31.5/25.1 (BP=0.966, ratio=0.966, hyp_len=3464, ref_len=3585)


## 8. Hyperparameter tuning

Glyph units scored lower than word units, so tuning uses word units. Each run changes the base Transformer (2 layers, emb 256, dropout 0.1).

- `tf-small`: 1 layer, emb 128, ffn 256, 2 heads
- `tf-drop`: dropout 0.3
- `tf-deep`: 3 layers, dropout 0.3

In [14]:
TUNE = ("tf-small", "tf-drop", "tf-deep")
for d in ("my-hi", "hi-my"):
    for m in TUNE:
        print(best_valid(f"{m}.word.{d}"))

tf-small.word.my-hi    best valid BLEU  30.03 at epoch 55, update 7200 (last epoch 61)
tf-drop.word.my-hi     best valid BLEU  34.49 at epoch 52, update 6800 (last epoch 61)
tf-deep.word.my-hi     best valid BLEU  32.38 at epoch 55, update 7200 (last epoch 61)
tf-small.word.hi-my    best valid BLEU  43.61 at epoch 58, update 7600 (last epoch 61)
tf-drop.word.hi-my     best valid BLEU  48.31 at epoch 61, update 7980 (last epoch 61)
tf-deep.word.hi-my     best valid BLEU  47.74 at epoch 61, update 7980 (last epoch 61)


In [15]:
for d in ("my-hi", "hi-my"):
    for m in TUNE:
        !python3 nmt.py test {m} {d}

tf-small.word my-hi: BLEU = 28.16, 55.6/31.8/21.5/17.2 (BP=0.990, ratio=0.990, hyp_len=1558, ref_len=1573)


tf-drop.word my-hi: BLEU = 32.79, 59.6/36.5/26.6/22.7 (BP=0.970, ratio=0.970, hyp_len=1526, ref_len=1573)


tf-deep.word my-hi: BLEU = 31.24, 56.8/33.9/24.7/20.1 (BP=0.999, ratio=0.999, hyp_len=1571, ref_len=1573)


tf-small.word hi-my: BLEU = 42.85, 70.5/51.8/37.7/29.5 (BP=0.955, ratio=0.956, hyp_len=3427, ref_len=3585)


tf-drop.word hi-my: BLEU = 46.71, 73.9/56.0/42.5/34.4 (BP=0.941, ratio=0.943, hyp_len=3381, ref_len=3585)


tf-deep.word hi-my: BLEU = 45.66, 72.0/54.2/40.7/32.5 (BP=0.958, ratio=0.959, hyp_len=3438, ref_len=3585)


## 9. Ensemble

Marian decodes with several models at once. Models must share the same vocab.

In [16]:
for d in ("my-hi", "hi-my"):
    !python3 nmt.py ensemble {d} s2s tf --hi word
    !python3 nmt.py ensemble {d} s2s tf --hi char
    !python3 nmt.py ensemble {d} tf tf-drop tf-deep
    !python3 nmt.py ensemble {d} s2s tf tf-drop tf-deep

ensemble(s2s+tf).word my-hi: BLEU = 38.61, 63.1/41.9/32.6/29.3 (BP=0.968, ratio=0.969, hyp_len=1524, ref_len=1573)


ensemble(s2s+tf).char my-hi: BLEU = 31.73, 59.0/35.1/24.7/19.9 (BP=1.000, ratio=1.007, hyp_len=1584, ref_len=1573)


ensemble(tf+tf-drop+tf-deep).word my-hi: BLEU = 34.88, 61.0/38.9/29.3/23.7 (BP=0.974, ratio=0.974, hyp_len=1532, ref_len=1573)


ensemble(s2s+tf+tf-drop+tf-deep).word my-hi: BLEU = 36.72, 62.4/40.6/31.1/25.6 (BP=0.975, ratio=0.975, hyp_len=1534, ref_len=1573)


ensemble(s2s+tf).word hi-my: BLEU = 51.69, 74.2/57.9/45.3/37.8 (BP=0.992, ratio=0.992, hyp_len=3558, ref_len=3585)


ensemble(s2s+tf).char hi-my: BLEU = 45.02, 68.9/51.9/39.5/33.0 (BP=0.969, ratio=0.969, hyp_len=3475, ref_len=3585)


ensemble(tf+tf-drop+tf-deep).word hi-my: BLEU = 49.60, 75.3/57.7/44.7/37.2 (BP=0.957, ratio=0.958, hyp_len=3433, ref_len=3585)


ensemble(s2s+tf+tf-drop+tf-deep).word hi-my: BLEU = 51.93, 75.6/58.9/46.4/39.2 (BP=0.973, ratio=0.973, hyp_len=3489, ref_len=3585)


## 10. Results

In [17]:
from IPython.display import Markdown

table = !python3 nmt.py report
Markdown("\n".join(table))

| Model | my-hi | hi-my |
| --- | ---: | ---: |
| s2s.word | 33.78 | 48.71 |
| s2s.char | 27.23 | 44.75 |
| tf.word | 33.66 | 49.65 |
| tf.char | 26.59 | 37.06 |
| tf-small.word | 28.16 | 42.85 |
| tf-drop.word | 32.79 | 46.71 |
| tf-deep.word | 31.24 | 45.66 |
| ensemble(s2s+tf).word | 38.61 | 51.69 |
| ensemble(s2s+tf).char | 31.73 | 45.02 |
| ensemble(tf+tf-drop+tf-deep).word | 34.88 | 49.60 |
| ensemble(s2s+tf+tf-drop+tf-deep).word | 36.72 | 51.93 |

In [18]:
!cat results/bleu.tsv

s2s.word	my-hi	BLEU = 33.78, 59.4/35.8/26.2/25.2 (BP=0.982, ratio=0.982, hyp_len=1545, ref_len=1573)
s2s.char	my-hi	BLEU = 27.23, 56.1/32.3/21.2/14.3 (BP=1.000, ratio=1.028, hyp_len=1617, ref_len=1573)
s2s.word	hi-my	BLEU = 48.71, 71.6/54.5/41.9/34.7 (BP=0.998, ratio=0.998, hyp_len=3579, ref_len=3585)
s2s.char	hi-my	BLEU = 44.75, 69.3/51.8/39.2/32.1 (BP=0.971, ratio=0.971, hyp_len=3481, ref_len=3585)
tf.word	my-hi	BLEU = 33.66, 60.2/37.2/28.1/24.2 (BP=0.958, ratio=0.959, hyp_len=1509, ref_len=1573)
tf.char	my-hi	BLEU = 26.59, 54.7/30.0/19.5/15.6 (BP=1.000, ratio=1.020, hyp_len=1604, ref_len=1573)
tf.word	hi-my	BLEU = 49.65, 73.8/56.9/43.9/36.3 (BP=0.976, ratio=0.977, hyp_len=3501, ref_len=3585)
tf.char	hi-my	BLEU = 37.06, 62.4/44.0/31.5/25.1 (BP=0.966, ratio=0.966, hyp_len=3464, ref_len=3585)
tf-small.word	my-hi	BLEU = 28.16, 55.6/31.8/21.5/17.2 (BP=0.990, ratio=0.990, hyp_len=1558, ref_len=1573)

tf-deep.word	my-hi	BLEU = 31.24, 56.8/33.9/24.7/20.1 (BP=0.999, ratio=0.999, hyp_len=1571

## 11. Sample translations

In [19]:
def show(src_file, ref_file, hyp_file, n=8):
    rows = zip(*(Path(f).read_text().splitlines() for f in (src_file, ref_file, hyp_file)))
    for src, ref, hyp in list(rows)[:n]:
        print("SRC", src)
        print("REF", ref)
        print("HYP", hyp, "\n")


show("myhi_data/test.my", "data/test.hi", "models/ensemble.word.my-hi/test.s2s+tf.txt")

SRC အချစ်က နာကျင်စေတယ်။
REF 𓂋𓐍𓎼𓂝 𓊪𓎡𓋴𓈎𓈙
HYP 𓂋𓐍𓎼𓂝 𓂨𓀀𓎡 𓊪𓎡𓋴𓈎 

SRC ကျွန်မမှာ အစားအစာမရှိဘူး။
REF 𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌
HYP 𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌 

SRC ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။
REF 𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝
HYP 𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝 

SRC ခင်ဗျားက သတိထားတတ်သူပါ။
REF 𓂨𓀀𓎡𓋴𓂝 𓐍𓇋𓈙𓂝𓋴𓎼𓄿𓎛𓈎
HYP 𓂨𓀀𓎡𓋴𓂝 𓏭𓄿𓎡𓈎𓆒𓀀𓎡𓈙 

SRC အသင်းက စောင့်နေခဲ့တယ်။
REF 𓈎𓊪𓂝 𓈎𓂝𓄿𓉔 𓏏𓄿𓆑𓈎𓂝𓇌
HYP 𓈎𓊪𓂝 𓈎𓂝𓄿𓉔 𓏏𓄿𓈙 𓏏𓄿𓆑𓈎𓆑𓎛𓃀 

SRC ခရီးကောင်းပါ
REF 𓊪𓄿𓎼𓂝 𓄿 𓎛𓆑𓏭𓂝 𓈎𓋴𓆑𓄡
HYP 𓊪𓄿𓎼𓂝 𓄿 𓈙𓄿𓅱𓂝 𓈎𓋴𓆑𓄡 

SRC ဘယ်လိုပဲဖြစ်ဖြစ် လုပ်ခဲ့တယ်။
REF 𓆑 𓇌𓆑𓇌 𓆑𓈎 𓄿𓎛𓂧𓏏𓄿𓂧
HYP 𓆑 𓇌𓆑𓇌 𓆑𓈎 𓇋𓂝 

SRC အခန်းက မှောင်နေတယ်။
REF 𓈎𓊪𓂝 𓋴𓐍𓐍𓉔 𓆑𓈙 𓇌𓄿𓋴𓈖
HYP 𓈎𓊪𓂝 𓋴𓐍𓐍𓉔 𓆑𓈙 𓇌𓄿𓋴𓈖 



In [20]:
show("myhi_data/test.hi", "data/test.my", "models/ensemble.word.hi-my/test.s2s+tf+tf-drop+tf-deep.txt")

SRC 𓂋𓐍𓎼𓂝 𓊪𓎡𓋴𓈎𓈙
REF အ ချစ် က နာ ကျင် စေ တယ် ။
HYP နာ ကျင် တာ ကို ကြိုက် တယ် ။ 

SRC 𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌
REF ကျွန် မ မှာ အ စား အ စာ မ ရှိ ဘူး ။
HYP ကျွန် မ မှာ အ စား အ စာ မ ရှိ ဘူး ။ 

SRC 𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝
REF ဘယ် သူ မှ ကျွန် မ ကို မ မြင် ခဲ့ ဘူး ။
HYP ဘယ် သူ မှ ကျွန် မ ကို မ မြင် ခဲ့ ဘူး ။ 

SRC 𓂨𓀀𓎡𓋴𓂝 𓐍𓇋𓈙𓂝𓋴𓎼𓄿𓎛𓈎
REF ခင် ဗျား က သ တိ ထား တတ် သူ ပါ ။
HYP မင်း က စောင့် ကြည့် နေ သူ ပါ ။ 

SRC 𓈎𓊪𓂝 𓈎𓂝𓄿𓉔 𓏏𓄿𓆑𓈎𓂝𓇌
REF အ သင်း က စောင့် နေ ခဲ့ တယ် ။
HYP အ သင်း က စောင့် နေ ခဲ့ တယ် ။ 

SRC 𓊪𓄿𓎼𓂝 𓄿 𓎛𓆑𓏭𓂝 𓈎𓋴𓆑𓄡
REF ခ ရီး ကောင်း ပါ
HYP ခ ရီး ကောင်း ပါ ဦး ။ 

SRC 𓆑 𓇌𓆑𓇌 𓆑𓈎 𓄿𓎛𓂧𓏏𓄿𓂧
REF ဘယ် လို ပဲ ဖြစ် ဖြစ် လုပ် ခဲ့ တယ် ။
HYP ဘယ် လို ဖြစ် ခဲ့ လဲ ။ 

SRC 𓈎𓊪𓂝 𓋴𓐍𓐍𓉔 𓆑𓈙 𓇌𓄿𓋴𓈖
REF အ ခန်း က မှောင် နေ တယ် ။
HYP အ ခန်း က အ မှောင် နေ တယ် ။ 



## 12. Summary

- Best my-hi: ensemble of s2s + Transformer, BLEU 38.61.
- Best hi-my: ensemble of s2s + three Transformers, BLEU 51.93.
- Single models: s2s and the base Transformer are close. s2s 33.78 / 48.71, Transformer 33.66 / 49.65.
- Word units for Hiero beat glyph units in every case. Spelling a Hiero word glyph by glyph is harder for this small data.
- Tuning did not beat the base Transformer. The small model underfits. On hi-my, the dropout and deeper models were still improving when training stopped at 60 epochs.
- Ensembles help most when the models differ. s2s + Transformer gains about 5 BLEU on my-hi. Three Transformers together gain about 1.
- hi-my scores are higher than my-hi. Myanmar is scored on syllables, which are easier to match than whole Hiero words.